In [2]:
!pip install fastf1

   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
    --------------------------------------- 0.3/11.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.1 MB 2.9 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.1 MB 2.2 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/11.1 MB 2.8 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/11.1 MB 3.2 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/11.1 MB 3.6 MB/s eta 0:00:03
   ----------------- ---------------------- 5.0/11.1 MB 3.6 MB/s eta 0:00:02
   ---------------------- ----------------- 6.3/11.1 MB 4.0 MB/s eta 0:00:02
   --------------------------- ------------ 7.6/11.1 MB 4.3 MB/s eta 0:00:01
   ------------------------------ --------- 8.4/11.1 MB 4.3 MB/s eta 0:00:01
   --------------------------------- ------ 9.2/11.1 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 9.7/11.1 MB 4.0 MB/s eta 0:00:01
   ----------

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os

# ============================================================
# SETTINGS
# ============================================================

START_YEAR = 2018
END_YEAR = 2026

CACHE_DIR = "f1cache"
OUTPUT_DIR = "F1_Data"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

fastf1.Cache.enable_cache(CACHE_DIR)


# ============================================================
# ONE DATASET
# ============================================================

ALL_DATA = []


# ============================================================
# LOOP YEARS
# ============================================================

for year in range(START_YEAR, END_YEAR + 1):

    print("\n" + "=" * 80)
    print(f"SEASON {year}")
    print("=" * 80)

    try:
        schedule = fastf1.get_event_schedule(year)

    except Exception as e:
        print(f"Schedule error: {e}")
        continue


    # ========================================================
    # LOOP RACES
    # ========================================================

    for _, event in schedule.iterrows():

        round_number = event["RoundNumber"]
        race_name = event["EventName"]

        # Skip testing / non-race events
        if pd.isna(round_number) or round_number == 0:
            continue

        round_number = int(round_number)

        print(
            f"\n{year} | Round {round_number} | {race_name}"
        )


        try:

            # =================================================
            # LOAD RACE
            # =================================================

            session = fastf1.get_session(
                year,
                round_number,
                "R"
            )

            session.load(
                laps=True,
                telemetry=False,
                weather=True,
                messages=False
            )


            # =================================================
            # RESULTS
            # =================================================

            results = session.results.copy()

            results["Position"] = pd.to_numeric(
                results["Position"],
                errors="coerce"
            )

            results["GridPosition"] = pd.to_numeric(
                results["GridPosition"],
                errors="coerce"
            )

            results["Points"] = pd.to_numeric(
                results["Points"],
                errors="coerce"
            )


            # =================================================
            # WEATHER
            # ONLY WEATHER TYPE IS KEPT
            # =================================================

            weather = session.weather_data

            if (
                weather is not None
                and not weather.empty
                and "Rainfall" in weather.columns
            ):

                rain = weather["Rainfall"].fillna(False)

                weather_type = (
                    "Wet"
                    if rain.astype(bool).any()
                    else "Dry"
                )

            else:

                weather_type = "Unknown"


            # =================================================
            # ALL LAPS
            # =================================================

            laps = session.laps.copy()


            # =================================================
            # EVERY DRIVER
            # =================================================

            for _, result in results.iterrows():

                driver = result["Abbreviation"]
                driver_name = result["FullName"]
                team = result["TeamName"]

                start_position = result["GridPosition"]
                finish_position = result["Position"]

                race_points = result["Points"]
                status = result["Status"]


                # =================================================
                # PODIUM
                # =================================================

                podium = (
                    "Yes"
                    if finish_position in [1, 2, 3]
                    else "No"
                )


                # =================================================
                # DRIVER LAPS
                # =================================================

                driver_laps = laps[
                    laps["Driver"] == driver
                ].copy()


                # =================================================
                # NO LAP DATA
                # =================================================

                if driver_laps.empty:

                    row = {

                        "Season": year,
                        "Round": round_number,
                        "Race": race_name,

                        "Driver": driver_name,
                        "Abbreviation": driver,
                        "Team": team,

                        "StartPosition": start_position,
                        "FinishPosition": finish_position,

                        "RacePoints": race_points,
                        "Podium": podium,
                        "Status": status,

                        "TotalLaps": np.nan,

                        "FastestLap": np.nan,
                        "AverageLapTime": np.nan,

                        "FastestS1": np.nan,
                        "FastestS2": np.nan,
                        "FastestS3": np.nan,

                        "PitStopLaps": "",

                        "Weather": weather_type
                    }

                    ALL_DATA.append(row)

                    continue


                # =================================================
                # TOTAL LAPS
                # =================================================

                total_laps = (
                    driver_laps["LapNumber"].max()
                )


                # =================================================
                # ACCURATE LAPS
                # NOT SAVED AS A COLUMN
                # ONLY USED FOR CALCULATIONS
                # =================================================

                if "IsAccurate" in driver_laps.columns:

                    accurate_laps = driver_laps[
                        driver_laps["IsAccurate"] == True
                    ].copy()

                else:

                    accurate_laps = driver_laps.copy()


                # =================================================
                # SECTORS
                # =================================================

                s1 = accurate_laps[
                    "Sector1Time"
                ].dropna()

                s2 = accurate_laps[
                    "Sector2Time"
                ].dropna()

                s3 = accurate_laps[
                    "Sector3Time"
                ].dropna()


                # =================================================
                # FASTEST SECTOR 1
                # =================================================

                if not s1.empty:

                    fastest_s1 = (
                        s1.min().total_seconds()
                    )

                else:

                    fastest_s1 = np.nan


                # =================================================
                # FASTEST SECTOR 2
                # =================================================

                if not s2.empty:

                    fastest_s2 = (
                        s2.min().total_seconds()
                    )

                else:

                    fastest_s2 = np.nan


                # =================================================
                # FASTEST SECTOR 3
                # =================================================

                if not s3.empty:

                    fastest_s3 = (
                        s3.min().total_seconds()
                    )

                else:

                    fastest_s3 = np.nan


                # =================================================
                # LAP TIMES
                # =================================================

                lap_times = accurate_laps[
                    "LapTime"
                ].dropna()


                if not lap_times.empty:

                    fastest_lap = (
                        lap_times.min()
                        .total_seconds()
                    )

                    average_lap = (
                        lap_times.mean()
                        .total_seconds()
                    )

                else:

                    fastest_lap = np.nan
                    average_lap = np.nan


                # =================================================
                # PIT STOP LAPS
                # PIT STOP COUNT REMOVED
                # =================================================

                if "PitInTime" in driver_laps.columns:

                    pit_rows = driver_laps[
                        driver_laps["PitInTime"].notna()
                    ]

                    pit_laps = (
                        pit_rows["LapNumber"]
                        .dropna()
                        .astype(int)
                        .tolist()
                    )

                else:

                    pit_laps = []


                pit_laps_text = ",".join(
                    map(str, pit_laps)
                )


                # =================================================
                # CREATE ONE ROW PER DRIVER PER RACE
                # =================================================

                row = {

                    # -------------------------------
                    # RACE
                    # -------------------------------

                    "Season": year,
                    "Round": round_number,
                    "Race": race_name,

                    # -------------------------------
                    # DRIVER
                    # -------------------------------

                    "Driver": driver_name,
                    "Abbreviation": driver,
                    "Team": team,

                    # -------------------------------
                    # RESULT
                    # -------------------------------

                    "StartPosition": start_position,
                    "FinishPosition": finish_position,

                    "RacePoints": race_points,
                    "Podium": podium,
                    "Status": status,

                    # -------------------------------
                    # LAPS
                    # -------------------------------

                    "TotalLaps": total_laps,

                    # -------------------------------
                    # LAP PERFORMANCE
                    # -------------------------------

                    "FastestLap": fastest_lap,
                    "AverageLapTime": average_lap,

                    # -------------------------------
                    # SECTORS
                    # -------------------------------

                    "FastestS1": fastest_s1,
                    "FastestS2": fastest_s2,
                    "FastestS3": fastest_s3,

                    # -------------------------------
                    # PIT STOP LAPS
                    # -------------------------------

                    "PitStopLaps": pit_laps_text,

                    # -------------------------------
                    # WEATHER
                    # -------------------------------

                    "Weather": weather_type
                }


                ALL_DATA.append(row)


            print("   ✓ COMPLETED")


        except Exception as e:

            print(
                f"   ✗ SKIPPED: {race_name}"
            )

            print(
                f"      {type(e).__name__}: {e}"
            )

            continue


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(ALL_DATA)


# ============================================================
# SORT DATA
# ============================================================

if not df.empty:

    df = df.sort_values(
        [
            "Season",
            "Round",
            "FinishPosition",
            "Driver"
        ]
    ).reset_index(drop=True)


# ============================================================
# SAVE ONE CSV
# ============================================================

OUTPUT_FILE = (
    f"{OUTPUT_DIR}/"
    f"F1_2020_2026_RACE_DATA.csv"
)


df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 80)
print("F1 DATASET COMPLETED")
print("=" * 80)

print(
    f"Total rows: {len(df):,}"
)

print(
    f"Total columns: {len(df.columns)}"
)

print("\nColumns:")

for column in df.columns:
    print(f" - {column}")

print("\nCSV FILE:")

print(OUTPUT_FILE)

print("=" * 80)


SEASON 2018


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info



2018 | Round 1 | Australian Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


   ✓ COMPLETED

2018 | Round 2 | Bahrain Grand Prix


...

   ✗ SKIPPED: São Paulo Grand Prix
      RateLimitExceededError: any API: 500 calls/h

2026 | Round 21 | Las Vegas Grand Prix


core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


   ✗ SKIPPED: Las Vegas Grand Prix
      RateLimitExceededError: any API: 500 calls/h

2026 | Round 22 | Qatar Grand Prix


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


   ✗ SKIPPED: Qatar Grand Prix
      RateLimitExceededError: any API: 500 calls/h

2026 | Round 23 | Abu Dhabi Grand Prix
   ✗ SKIPPED: Abu Dhabi Grand Prix
      RateLimitExceededError: any API: 500 calls/h


F1 DATASET COMPLETED
Total rows: 3,746
Total columns: 19

Columns:
 - Season
 - Round
 - Race
 - Driver
 - Abbreviation
 - Team
 - StartPosition
 - FinishPosition
 - RacePoints
 - Podium
 - Status
 - TotalLaps
 - FastestLap
 - AverageLapTime
 - FastestS1
 - FastestS2
 - FastestS3
 - PitStopLaps
 - Weather

CSV FILE:
F1_Data/F1_2020_2026_RACE_DATA.csv
